# 🪶 Automated Feather Measurement and Analysis Pipeline
This notebook shows a small, complete run of the feather workflow.

Plain-language summary: choose one image, send it to remote workers, review the returned feather outputs, and then run biological summary steps. Heavy compute stays on the cluster; this notebook is the control panel.

The workflow mixes three ideas:
- **Zero-shot detection and segmentation** to find feathers without task-specific retraining.
- **Optional supervised training** (Roboflow + YOLO) when you want a custom model tuned to your own labels.
- **Biological interpretation steps** (BioCLIP + `pavo`) to summarize shape/texture cues and color-pattern structure.


## Step 1: Setup and Connection Check
This step sets your host, user, SSH key, and folder paths, then checks that the job queue is reachable.

If this cell finishes without error, the notebook is ready to send and monitor jobs.


In [ ]:
import os
import shlex
import socket
import subprocess
import time
from pathlib import Path
from urllib.parse import urlparse

from celery import Celery
from IPython.display import display
from PIL import Image

# --- Core remote connection settings ---
HEAD_HOST = os.getenv('FEATHER_HEAD_HOST', '10.0.0.148')
SSH_USER = os.getenv('FEATHER_SSH_USER', 'openteams')

# Try FEATHER_SSH_KEY first; if missing, try common key filenames.
key_candidates = [
    os.path.expanduser(os.getenv('FEATHER_SSH_KEY', '')),
    os.path.expanduser('~/.ssh/ubuntu-mac-openteams-admin'),
    os.path.expanduser('~/.ssh/ubuntu-mac-cluster_user-admin'),
    os.path.expanduser('~/.ssh/id_ed25519'),
    os.path.expanduser('~/.ssh/llama_watchdog_key'),
]
KEY_PATH = next((k for k in key_candidates if k and os.path.exists(k)), '')

# --- Remote folders used by the workers ---
REMOTE_REPO_DIR = os.getenv('FEATHER_REMOTE_REPO_DIR', '~/Feather_Molt_Project')
REMOTE_INPUT_DIR = os.getenv('FEATHER_REMOTE_INPUT_DIR', f'{REMOTE_REPO_DIR}/data/raw')
REMOTE_OUTPUT_DIR = os.getenv('FEATHER_REMOTE_OUTPUT_DIR', f'{REMOTE_REPO_DIR}/data/processed')
CLUSTER_HOSTS = [h.strip() for h in os.getenv('FEATHER_CLUSTER_HOSTS', '10.0.0.148,10.0.0.63,10.0.0.19,10.0.0.118').split(',') if h.strip()]

# --- Task queue settings (Celery + Redis) ---
BROKER_URL = os.getenv('BROKER_URL', f'redis://{HEAD_HOST}:6379/0')
RESULT_BACKEND = os.getenv('RESULT_BACKEND', '')
if RESULT_BACKEND:
    os.environ['RESULT_BACKEND'] = RESULT_BACKEND
os.environ['BROKER_URL'] = BROKER_URL


def assert_tcp_reachable(url: str, name: str, timeout: float = 1.5):
    """Quick socket check so failures are caught early with a clear message."""
    parsed = urlparse(url)
    host = parsed.hostname
    port = parsed.port or 6379
    if not host:
        raise RuntimeError(f'{name} URL is invalid: {url}')

    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
    except Exception as exc:
        raise RuntimeError(
            f'{name} not reachable at {host}:{port} from this notebook host. '
            f'Original error: {type(exc).__name__}: {exc}'
        ) from exc
    finally:
        s.close()


if os.getenv('FEATHER_SKIP_BROKER_CHECK', '0') == '1':
    print('Skipping broker reachability check (FEATHER_SKIP_BROKER_CHECK=1)')
else:
    print('Checking broker reachability...')
    timeout = float(os.getenv('FEATHER_SOCKET_TIMEOUT', '1.5'))
    assert_tcp_reachable(BROKER_URL, 'BROKER_URL', timeout=timeout)

# Use result backend only when explicitly provided.
backend = None
if RESULT_BACKEND:
    timeout = float(os.getenv('FEATHER_SOCKET_TIMEOUT', '1.5'))
    assert_tcp_reachable(RESULT_BACKEND, 'RESULT_BACKEND', timeout=timeout)
    backend = RESULT_BACKEND

celery_app = Celery('feather_pipeline', broker=BROKER_URL, backend=backend)

# Local folder for fetched preview files.
LOCAL_PREVIEW_DIR = Path('./.minimal_slice_preview').resolve()
LOCAL_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print('HEAD_HOST =', HEAD_HOST)
print('SSH_USER =', SSH_USER)
print('KEY_PATH =', KEY_PATH or '(none found; set FEATHER_SSH_KEY)')
print('CLUSTER_HOSTS =', CLUSTER_HOSTS)
print('BROKER_URL =', BROKER_URL)
print('RESULT_BACKEND =', RESULT_BACKEND or '(disabled)')
print('REMOTE_INPUT_DIR =', REMOTE_INPUT_DIR)
print('REMOTE_OUTPUT_DIR =', REMOTE_OUTPUT_DIR)
print('LOCAL_PREVIEW_DIR =', LOCAL_PREVIEW_DIR)


## Step 2: Remote File Helpers
This step defines helper functions to list and copy files on remote machines.

This lets you work directly with remote image folders without copying the full dataset to your notebook machine.


In [ ]:
def _ssh_base_cmd(host: str):
    """Build a strict SSH command that uses one key only."""
    cmd = [
        'ssh',
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'IdentitiesOnly=yes',
        '-o', 'IdentityAgent=none',
        '-o', 'BatchMode=yes',
        '-o', 'PreferredAuthentications=publickey',
    ]
    if not KEY_PATH:
        raise RuntimeError('No SSH key found. Set FEATHER_SSH_KEY to a readable private key path.')
    cmd += ['-i', KEY_PATH]
    cmd += [f'{SSH_USER}@{host}']
    return cmd


def _remote_shell_path(path: str) -> str:
    """Preserve remote $HOME expansion while handling spaces safely."""
    if path == '~':
        return '"$HOME"'
    if path.startswith('~/'):
        return '"$HOME/' + path[2:] + '"'
    return shlex.quote(path)


def ssh_lines(host: str, remote_cmd: str):
    """Run a remote command and return non-empty output lines."""
    out = subprocess.check_output([*_ssh_base_cmd(host), remote_cmd], text=True)
    return [line.strip() for line in out.splitlines() if line.strip()]




_REMOTE_HOME_CACHE = {}

def remote_home(host: str = HEAD_HOST) -> str:
    """Get remote $HOME once and cache it for later path expansion."""
    if host not in _REMOTE_HOME_CACHE:
        home = subprocess.check_output([*_ssh_base_cmd(host), 'printf %s "$HOME"'], text=True).strip()
        if not home:
            raise RuntimeError(f'Could not resolve remote home for {host}')
        _REMOTE_HOME_CACHE[host] = home
    return _REMOTE_HOME_CACHE[host]


def remote_runtime_path(path: str, host: str = HEAD_HOST) -> str:
    """Convert ~/... to an absolute path for worker-side Python APIs."""
    if path == '~':
        return remote_home(host)
    if path.startswith('~/'):
        return f"{remote_home(host)}/{path[2:]}"
    return path

def list_remote_images(input_host: str = HEAD_HOST):
    """List source images in the remote input folder."""
    d = _remote_shell_path(REMOTE_INPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f \\( -iname '*.jpg' -o -iname '*.jpeg' \\) | sort"
    return ssh_lines(input_host, cmd)


def list_remote_outputs(host: str):
    """List output images in a worker's processed folder."""
    d = _remote_shell_path(REMOTE_OUTPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f -iname '*.jpg' | sort"
    return ssh_lines(host, cmd)


def fetch_remote_file(host: str, remote_path: str, local_dir: Path):
    """Copy one remote file into a local directory."""
    local_dir.mkdir(parents=True, exist_ok=True)
    local_file = local_dir / Path(remote_path).name
    remote_cmd = f"cat {shlex.quote(remote_path)}"
    with open(local_file, 'wb') as f:
        subprocess.check_call([*_ssh_base_cmd(host), remote_cmd], stdout=f)


## Step 3: Send One Image for Processing
This step selects one source image and sends it to the workers.

What the workers do:
- **Grounding DINO** proposes feather regions from a text prompt.
- **SAM** turns those proposed regions into pixel-level feather masks.
- The pipeline then saves extracted feather crops and a box/mask overlay image.

Why this is useful:
- This is a **zero-shot** path, so it works without training a feather-specific detector first.
- It is fast to start and good for bootstrapping data review.


In [ ]:
# Get the full remote image list from the head node.
remote_images = list_remote_images(HEAD_HOST)
print(f'Found {len(remote_images)} remote images on {HEAD_HOST}.')
if not remote_images:
    raise RuntimeError('No remote images found. Check FEATHER_REMOTE_INPUT_DIR.')

# Pick one image for a quick minimal-slice run.
SAMPLE_INDEX = 0
image_path = remote_images[SAMPLE_INDEX]
source_stem = Path(image_path).stem
print('Selected remote source:', image_path)

# Resolve output path to an absolute path for worker-side Python.
output_dir_for_task = remote_runtime_path(REMOTE_OUTPUT_DIR, HEAD_HOST)
print('Worker output dir =', output_dir_for_task)

# Send the job to Celery workers. We poll output files in Step 4.
async_result = celery_app.send_task(
    'src.celery_tasks.process_image',
    args=[image_path, output_dir_for_task],
    ignore_result=True,
)
print('Task dispatched. ID:', async_result.id)


## Step 4: Pull Outputs and Review Them
This step checks workers for matching outputs, copies them into a local preview folder, and displays them inline.

You should see:
- a bounding-box preview image,
- one or more extracted feather images.


In [ ]:
# Output files share the source filename stem.
prefix = f"{source_stem}_"
matching = []

# Poll settings can be tuned with env vars.
WAIT_TIMEOUT_SECONDS = int(os.getenv('FEATHER_WAIT_TIMEOUT_SECONDS', '180'))
POLL_SECONDS = float(os.getenv('FEATHER_POLL_SECONDS', '3'))
deadline = time.time() + WAIT_TIMEOUT_SECONDS

# Keep checking each host until matching outputs appear or timeout is reached.
while time.time() < deadline and not matching:
    for host in CLUSTER_HOSTS:
        try:
            out_files = list_remote_outputs(host)
        except Exception as exc:
            print(f'Skipping host {host}: {exc}')
            continue

        host_matches = []
        for p in out_files:
            name = Path(p).name
            if name.startswith(prefix) and ('_Feather_' in name or '_BoundingBoxes' in name):
                host_matches.append(p)

        if host_matches:
            print(f'Found {len(host_matches)} segment(s) on {host}.')
            matching = [(host, p) for p in host_matches]
            break

    if not matching:
        print('Waiting for output files...')
        time.sleep(POLL_SECONDS)

if not matching:
    raise RuntimeError(
        f'No matching segment outputs found within {WAIT_TIMEOUT_SECONDS}s. '
        'Check worker status and remote output path.'
    )

# Fetch matched files into a local preview directory.
preview_dir = LOCAL_PREVIEW_DIR / source_stem
preview_dir.mkdir(parents=True, exist_ok=True)
for host, remote_file in matching:
    fetch_remote_file(host, remote_file, preview_dir)

# Display fetched files inline for quick manual review.
local_files = sorted(preview_dir.glob('*.jpg'))
print(f'Fetched {len(local_files)} files into {preview_dir}')

bbox_files = [f for f in local_files if '_BoundingBoxes' in f.name]
feather_files = [f for f in local_files if '_Feather_' in f.name]

for f in bbox_files:
    print('\n=== DETECTION BOUNDING BOXES ===')
    print(f.name)
    display(Image.open(f))

for f in feather_files:
    print('\n=== EXTRACTED FEATHER ===')
    print(f.name)
    display(Image.open(f))


## Optional Step 5: Train a Custom Model with Roboflow Data
Use this step if you want a custom model trained on your curated labels.

How this compares with Step 3:
- **Step 3 (Grounding DINO + SAM)** is zero-shot: no feather-specific training required.
- **This step (Roboflow + YOLO)** is supervised: you curate/correct labels, then train a model tuned to your dataset.

When to use which:
- Use zero-shot for quick start, discovery, and early data triage.
- Use Roboflow + YOLO when you want stable behavior at scale and tighter control over error modes.

Simple workflow:
1. Review outputs from Step 4 and keep strong examples.
2. Correct labels in Roboflow.
3. Export the dataset in YOLO format.
4. Run the training cell below.


In [ ]:
import os
from pathlib import Path

# Path to YOLO-format dataset exported from Roboflow.
DATASET_YAML = Path('../data/autolabel/dataset.yaml')

# Safety switch: training only runs when explicitly enabled.
RUN_ROBOFLOW_TRAINING = os.getenv('FEATHER_RUN_ROBOFLOW_TRAINING', '0') == '1'

if not RUN_ROBOFLOW_TRAINING:
    print('Skipping training (set FEATHER_RUN_ROBOFLOW_TRAINING=1 to enable).')
elif not DATASET_YAML.exists():
    print('Roboflow export not found at ../data/autolabel/dataset.yaml')
    print('Export your curated dataset (YOLO format) to ../data/autolabel/ and re-run this cell.')
else:
    try:
        from ultralytics import YOLO
    except Exception as exc:  # noqa: BLE001
        print(f'Ultralytics import failed: {exc}')
        print('Install with: pip install ultralytics')
    else:
        # Train a lightweight segmentation baseline from curated labels.
        model = YOLO('yolov12n-seg.pt')
        print('Starting YOLOv12 segmentation fine-tuning...')
        _ = model.train(
            data=str(DATASET_YAML),
            epochs=100,
            batch=16,
            device='mps',
            project='../models',
            name='yolov12_feather_seg',
            plots=True,
        )
        print('Training complete. Best weights at ../models/yolov12_feather_seg/weights/best.pt')


## Step 6: Biological Feature Summary (BioCLIP + Heatmap)
This step turns one feather image into a numeric feature vector and creates a heatmap showing which regions most influenced the model score.

Why the heatmap is useful:
- It helps verify the model is focusing on feather texture/pattern regions rather than background artifacts.
- It gives an interpretable QA signal when comparing runs or reviewing unusual cases.

How to read the output:
- **Feature vector (embedding):** compact numeric summary of visual morphology.
- **Heatmap overlay:** brighter regions indicate stronger influence on the similarity score.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Reuse one extracted feather crop from Step 4.
feather_files = [f for f in local_files if '_Feather_' in f.name]
if not feather_files:
    raise RuntimeError('No feather crops found. Run Step 4 first.')

crop_path = str(feather_files[0])
print('BioCLIP input:', crop_path)

BIOCLIP_READY = True
try:
    import torch
    import open_clip
    from PIL import Image
except Exception as exc:  # noqa: BLE001
    print('BioCLIP dependencies missing; skipping this step.')
    print('Install with: pip install open_clip_torch torch torchvision')
    print('Details:', exc)
    BIOCLIP_READY = False

if BIOCLIP_READY:
    try:
        # Use Apple GPU when available; otherwise CPU.
        device = 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'
        print('device =', device)

        model_name = os.getenv('FEATHER_BIOCLIP_MODEL', 'ViT-H-14')

        # Use tags known to exist in this environment by default.
        requested = os.getenv('FEATHER_BIOCLIP_PRETRAINED', '').strip()
        fallback_tags = ['dfn5b', 'metaclip_fullcc', 'metaclip_altogether', 'laion2b_s32b_b79k']
        tag_candidates = [requested] if requested else []
        tag_candidates.extend(t for t in fallback_tags if t not in tag_candidates)

        model = None
        preprocess = None
        loaded_tag = None
        last_exc = None

        for tag in tag_candidates:
            try:
                model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=tag)
                loaded_tag = tag
                break
            except Exception as exc:  # noqa: BLE001
                last_exc = exc
                continue

        if model is None or preprocess is None:
            raise RuntimeError(f'Could not load BioCLIP/OpenCLIP weights. Last error: {last_exc}')

        print('Loaded OpenCLIP weights:', loaded_tag)
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer(model_name)

        # Prepare one image and one simple text prompt.
        img_pil = Image.open(crop_path).convert('RGB')
        img_tensor = preprocess(img_pil).unsqueeze(0).to(device)
        img_tensor.requires_grad_(True)
        text = tokenizer(['bird feather morphology']).to(device)

        # Compute normalized text and image features.
        with torch.no_grad():
            text_features = model.encode_text(text)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        image_features = model.encode_image(img_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        score = (image_features @ text_features.T).squeeze()
        score.backward()

        # Embedding is the compact numeric summary vector.
        embedding = image_features.detach().cpu().numpy().reshape(-1)
        print('Embedding dimension:', embedding.shape[0])
        print('Text-image similarity score:', float(score.detach().cpu().item()))

        # Create a simple saliency map from image gradients.
        grad = img_tensor.grad.detach().abs().mean(dim=1).squeeze().cpu().numpy()
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        img_np = np.array(img_pil)
        heat = np.uint8(255 * grad)
        heat = np.array(Image.fromarray(heat).resize((img_np.shape[1], img_np.shape[0])))

        # Show the original crop and the heatmap overlay side by side.
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(img_np)
        plt.title('Feather Crop')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(img_np)
        plt.imshow(heat, alpha=0.45, cmap='inferno')
        plt.title('BioCLIP Saliency Overlay')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as exc:  # noqa: BLE001
        print('BioCLIP step failed; skipping and continuing notebook.')
        print('Details:', exc)


## Step 7: Color Pattern Summary in R (`pavo`)
This step sends one extracted feather to `pavo`, groups similar colors, and computes adjacency statistics.

How to read the outputs:
- **Classified image:** reduces many natural colors into a few dominant color groups.
- **Adjacency statistics (`adj_stats`):** summarize how often each color group borders another. This helps quantify pattern structure (for example, mottling or banding).

Together with Step 6, this gives both a machine-learning summary (embedding/heatmap) and a classic color-pattern summary (`pavo`).


In [ ]:
import subprocess
import os
import sys
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Reuse one extracted feather crop from Step 4.
feather_files = [f for f in local_files if "_Feather_" in f.name]
if not feather_files:
    raise RuntimeError("No feather crops found to analyze.")
crop_path = str(feather_files[0])

# Build a short R script for pavo classification + adjacency metrics.
r_script = f"""
suppressMessages(library(pavo))

cat('Analyzing segmented crop directly in R: {crop_path}\n')

# 1) Load image.
feather_img <- getimg('{crop_path}')

# 2) Group colors into three major classes.
classified_feather <- classify(feather_img, kcols = 3)

# 3) Export class matrix + class colors for plotting in Python.
write.table(attr(classified_feather, "classRGB"), "class_colors.csv", row.names=FALSE, col.names=FALSE, sep=",")
write.table(as.matrix(classified_feather), "class_matrix.csv", row.names=FALSE, col.names=FALSE, sep=",")

# 4) Print adjacency statistics.
adj_stats <- adjacent(classified_feather, xscale = 1)
print(adj_stats)
"""

env = os.environ.copy()
if sys.platform == "darwin":
    # Help R locate its framework install on macOS.
    env['R_HOME'] = "/Library/Frameworks/R.framework/Resources"
    env['PATH'] = env['R_HOME'] + "/bin:" + env.get('PATH', "")

print("Executing pavo R script...")
try:
    result = subprocess.run(['Rscript', '-e', r_script], capture_output=True, text=True, env=env, timeout=int(os.getenv('FEATHER_R_TIMEOUT_SECONDS', '90')))
    print(result.stdout)
    if result.stderr:
        print("R stderr:", result.stderr)
except FileNotFoundError:
    print("Rscript not found on this system. Skipping pavo analysis.")
except subprocess.TimeoutExpired:
    print('R pavo step timed out; skipping and continuing notebook.')

# Rebuild a clean side-by-side plot in Python.
if os.path.exists('class_matrix.csv') and os.path.exists('class_colors.csv'):
    img1 = cv2.imread(crop_path)[:, :, ::-1]  # BGR -> RGB

    matrix = np.loadtxt('class_matrix.csv', delimiter=',')
    colors = np.loadtxt('class_colors.csv', delimiter=',')  # RGB rows

    img2 = np.zeros((matrix.shape[0], matrix.shape[1], 3))
    for i in range(1, len(colors) + 1):
        img2[matrix == i] = colors[i - 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    ax1.imshow(img1)
    ax1.set_title("1. Segmented Crop", fontsize=14, pad=15)
    ax1.axis('off')

    ax2.imshow(img2)
    ax2.set_title("2. pavo k=3 Color Classification", fontsize=14, pad=15)
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

    # Remove temporary files.
    os.remove('class_matrix.csv')
    os.remove('class_colors.csv')
